# Part 1: Data Processing (~1 page).

## Task 1: Retrieve sample:
Your first task is to retrieve a sample of the FakeNewsCorpus from https://raw.githubusercontent.com/several27/FakeNewsCorpus/master/news_sample.csv to an external site.
and structure, process, clean it. You should follow the methodology you developed in Exercise 1.

In [8]:
import pandas as pd

def getgit(url):
    data = pd.read_csv(
        url,
        encoding="utf-8",
        engine="python"  
    )
    return data

fnc = getgit("https://raw.githubusercontent.com/several27/FakeNewsCorpus/master/news_sample.csv")

print(fnc.head())
print(fnc.shape)


   Unnamed: 0   id                domain        type  \
0           0  141               awm.com  unreliable   
1           1  256     beforeitsnews.com        fake   
2           2  700           cnnnext.com  unreliable   
3           3  768               awm.com  unreliable   
4           4  791  bipartisanreport.com   clickbait   

                                                 url  \
0  http://awm.com/church-congregation-brings-gift...   
1  http://beforeitsnews.com/awakening-start-here/...   
2  http://www.cnnnext.com/video/18526/never-hike-...   
3  http://awm.com/elusive-alien-of-the-sea-caught...   
4  http://bipartisanreport.com/2018/01/21/trumps-...   

                                             content  \
0  Sometimes the power of Christmas will make you...   
1  AWAKENING OF 12 STRANDS of DNA – “Reconnecting...   
2  Never Hike Alone: A Friday the 13th Fan Film U...   
3  When a rare shark was caught, scientists were ...   
4  Donald Trump has the unnerving ability to a

### Define method to clean text:

In [9]:
#clean text
def clean_text(text):
    
    text = text.apply(lambda x: x.str.lower() if x.dtype == 'object' else x) # all words must be lowercased 
    text = text.apply(lambda x: x.str.strip() if x.dtype == 'object' else x) # it should not contain multiple white spaces, tabs, or new lines
    text = text.apply(lambda x: x.str.replace(r"\bhttps?://[^\s]+|www\.[^\s]+", "<URL>", regex = True) if x.dtype == 'object' else x)
    text = text.apply(lambda x: x.str.replace(r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[A-Za-z]{2,}", "<EMAIL>", regex = True) if x.dtype == 'object' else x)
    text = text.apply(lambda x: x.str.replace(r"\b(\d{4}-\d{2}-\d{2}|\d{1,2}[/-]\d{1,2}[/-]\d{2,4})\b", "<DATE>", regex = True) if x.dtype == 'object' else x)
    text = text.apply(lambda x: x.str.replace(r"\d+", "<NUM>", regex = True) if x.dtype == 'object' else x)    
    
    return text

text = clean_text(fnc)

### Import required packages, and clean text:

In [10]:
import nltk
import numpy as np
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
nltk.download('punkt_tab')
nltk.download('stopwords')

if isinstance(text, pd.DataFrame):
    one_string = "\n".join(text.astype(str).apply(lambda row: " ".join(row.dropna().tolist()), axis=1).tolist()
    )

elif isinstance(text, list):
    one_string = " ".join(str(x) for x in text if x is not None)


[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Nickl\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Nickl\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


### Tokenize:
### What and Why it's used:
nltk is used to tokenzie as it's efficient and strongly hinted at by the task description.

In [11]:
tokens = word_tokenize(one_string)

token_count = len(tokens)
vocab_size = len(set(tokens))

print(f"Token count: {token_count}.")
print(f"Vocabulary size: {vocab_size}.")

Token count: 233159.
Vocabulary size: 18034.


### Remove Stopwords, and compute size:
### What and Why it's used:
nltk is used to remove stopwords as it's efficient and strongly hinted at by the task description.

In [12]:
stop_words = set(stopwords.words("english"))
filtered_tokens = [word for word in tokens if word.lower() not in stop_words]

#filtered_text = " ".join(filtered_tokens) 

vocab_nostop = len(set(filtered_tokens))

#compute size of vocab:
print(f"Size of vocabulary after removing stopwords: {vocab_nostop}.")

#Compute the reduction rate of the vocabulary size after removing stopwords:
print(f"Reduction in size after removing stopwords: {((vocab_size-vocab_nostop)/vocab_size)*100:.3f} %.")

Size of vocabulary after removing stopwords: 17888.
Reduction in size after removing stopwords: 0.810 %.


### Stemming and compute size of vocabolary:
### What and Why it's used:
nltk PorterStemmer, because why not.

In [13]:
from nltk.stem import PorterStemmer

ps = PorterStemmer()

stemmed_tokens = [ps.stem(word) for word in filtered_tokens]

vocab_stem = len(set(stemmed_tokens))

#Compute size of vocabulary:
print(f"Vocabulary size after stemming: {vocab_stem}.")

#Compute the reduction rate of the vocabulary size after stemming:
print(f"Reduction in size after removing word variations: {((vocab_nostop - vocab_stem) / vocab_nostop)*100:.3f} %.")

Vocabulary size after stemming: 12669.
Reduction in size after removing word variations: 29.176 %.
